# Coffee Roasting Classification using TensorFlow/Keras

This notebook builds a neural network to classify coffee roasting quality (Good or Bad) based on temperature and roasting duration.

## Project Overview
- **Objective**: Binary classification of coffee roasting quality
- **Features**: Temperature (°C) and Duration (minutes)
- **Target**: 1 = Good roasting, 0 = Bad roasting
- **Dataset**: 200 samples from Kaggle

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras import Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load and Explore Data

In [ ]:
# Download dataset from Kaggle
import kagglehub
path = kagglehub.dataset_download("youssefahmed2612/coffee-roasting")
print(f"Dataset downloaded to: {path}")

In [ ]:
import os
# Load the data
data = pd.read_csv(os.path.join(path, 'CoffeeRoasting.csv'))
print(f"Dataset shape: {data.shape}")
print(f"\nColumn names: {data.columns.tolist()}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
print(data.head())
print(f"\nDataset info:")
print(data.info())
print(f"\nStatistical summary:")
print(data.describe())

In [ ]:
# Check for missing values and class distribution
print(f"Missing values:\n{data.isnull().sum()}")
print(f"\nClass distribution:")
print(data['Target'].value_counts())
print(f"\nClass proportions:")
print(data['Target'].value_counts(normalize=True))

## 3. Data Preprocessing

In [ ]:
# Separate features and target
X = data[['Temprature', 'Duration']].values
y = data['Target'].values.reshape(-1, 1)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFirst 5 feature samples:\n{X[:5]}")
print(f"\nFirst 5 target samples:\n{y[:5]}")

In [ ]:
# Feature normalization
print("Before normalization:")
print(f"Temperature - Max: {X[:,0].max():.2f}, Min: {X[:,0].min():.2f}, Mean: {X[:,0].mean():.2f}")
print(f"Duration    - Max: {X[:,1].max():.2f}, Min: {X[:,1].min():.2f}, Mean: {X[:,1].mean():.2f}")

# Use Keras Normalization layer
norm_layer = tf.keras.layers.Normalization(axis=-1)
norm_layer.adapt(X)  # learns mean and variance
X_normalized = norm_layer(X).numpy()

print(f"\nAfter normalization:")
print(f"Temperature - Max: {X_normalized[:,0].max():.2f}, Min: {X_normalized[:,0].min():.2f}, Mean: {X_normalized[:,0].mean():.4f}")
print(f"Duration    - Max: {X_normalized[:,1].max():.2f}, Min: {X_normalized[:,1].min():.2f}, Mean: {X_normalized[:,1].mean():.4f}")

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42, stratify=y
)

# Further split training data for validation (75-25 of training data = 60-20-20 overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution: {np.bincount(y_train.flatten())}")
print(f"Validation set class distribution: {np.bincount(y_val.flatten())}")
print(f"Test set class distribution: {np.bincount(y_test.flatten())}")

## 4. Build Neural Network Model

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Build the model with improved architecture
model = Sequential([
    # Input layer with normalization
    tf.keras.Input(shape=(2,)),
    norm_layer,
    
    # Hidden layers with ReLU activation for better gradient flow
    Dense(64, activation='relu', name='hidden_1'),
    Dropout(0.3),  # Dropout for regularization
    
    Dense(32, activation='relu', name='hidden_2'),
    Dropout(0.2),
    
    Dense(16, activation='relu', name='hidden_3'),
    
    # Output layer with sigmoid for binary classification
    Dense(1, activation='sigmoid', name='output')
])

print("Model created successfully!")

In [ ]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

# Display model architecture
print("\nModel Architecture:")
model.summary()

## 5. Train the Model

In [ ]:
# Define callbacks for better training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.00001,
    verbose=1
)

# Train the model
print("Training the model...")
history = model.fit(
    X_train, y_train,
    batch_size=16,
    epochs=200,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

print(f"\nTraining completed!")
print(f"Total epochs trained: {len(history.history['loss'])}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss Over Epochs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy Over Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model Evaluation

In [ ]:
# Evaluate on training, validation, and test sets
train_loss, train_acc, train_precision, train_recall = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc, val_precision, val_recall = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_acc, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)

print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\n{'Metric':<20} {'Training':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Loss':<20} {train_loss:<15.4f} {val_loss:<15.4f} {test_loss:<15.4f}")
print(f"{'Accuracy':<20} {train_acc:<15.4f} {val_acc:<15.4f} {test_acc:<15.4f}")
print(f"{'Precision':<20} {train_precision:<15.4f} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<20} {train_recall:<15.4f} {val_recall:<15.4f} {test_recall:<15.4f}")
print("="*60)

In [ ]:
# Make predictions on test set
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Calculate additional metrics
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print(f"\nAdditional Metrics on Test Set:")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix (Test Set):")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")

print("\n" + "="*60)
print("Classification Report (Test Set):")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Bad Roasting', 'Good Roasting']))

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Bad Roasting', 'Good Roasting'],
            yticklabels=['Bad Roasting', 'Good Roasting'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 7. Make Predictions on New Data

In [ ]:
# Example: Make predictions on new coffee roasting conditions
new_roasts = np.array([
    [200, 12.5],  # Example 1: 200°C, 12.5 minutes
    [250, 14.0],  # Example 2: 250°C, 14 minutes
    [175, 11.8],  # Example 3: 175°C, 11.8 minutes
    [280, 13.5],  # Example 4: 280°C, 13.5 minutes
])

# Normalize the new data
new_roasts_normalized = norm_layer(new_roasts).numpy()

# Make predictions
predictions = model.predict(new_roasts_normalized, verbose=0)
predicted_classes = (predictions >= 0.5).astype(int)

print("\n" + "="*70)
print("PREDICTIONS ON NEW COFFEE ROASTING CONDITIONS")
print("="*70)
print(f"{'Temperature (°C)':<20} {'Duration (min)':<20} {'Probability':<15} {'Prediction':<15}")
print("-"*70)

for i, (temp, duration) in enumerate(new_roasts):
    prob = predictions[i][0]
    prediction = "Good Roasting ✓" if predicted_classes[i][0] == 1 else "Bad Roasting ✗"
    print(f"{temp:<20.1f} {duration:<20.1f} {prob:<15.4f} {prediction:<15}")
print("="*70)

## 8. Decision Boundary Visualization

In [ ]:
# Create a mesh for decision boundary visualization
temp_min, temp_max = X[:, 0].min() - 10, X[:, 0].max() + 10
duration_min, duration_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(temp_min, temp_max, 100),
    np.linspace(duration_min, duration_max, 100)
)

# Prepare data for prediction
mesh_data = np.c_[xx.ravel(), yy.ravel()]
mesh_data_normalized = norm_layer(mesh_data).numpy()
Z = model.predict(mesh_data_normalized, verbose=0)
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(12, 8))

# Denormalize test data for plotting
X_test_orig = X[data.sample(n=len(X_test), random_state=42).index]

# Plot decision boundary
contour = plt.contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

# Plot test data points
scatter = plt.scatter(X[:, 0], X[:, 1], c=y.flatten(), cmap='RdYlGn', 
                      edgecolors='black', s=100, alpha=0.7, linewidths=1.5)

plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Duration (minutes)', fontsize=12)
plt.title('Coffee Roasting Decision Boundary', fontsize=14, fontweight='bold')
plt.colorbar(contour, label='Probability of Good Roasting')
cbar = plt.colorbar(scatter, label='Actual Class')
cbar.set_ticks([0.25, 0.75])
cbar.set_ticklabels(['Bad', 'Good'])
plt.tight_layout()
plt.show()

## 9. Model Summary and Key Insights

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY AND KEY INSIGHTS")
print("="*70)

print(f"\n1. DATASET INFORMATION:")
print(f"   - Total samples: {len(data)}")
print(f"   - Training samples: {len(X_train)} (60%)")
print(f"   - Validation samples: {len(X_val)} (20%)")
print(f"   - Test samples: {len(X_test)} (20%)")
print(f"   - Features: Temperature (°C) and Duration (minutes)")
print(f"   - Class balance: {np.bincount(y.flatten())[0]} Bad, {np.bincount(y.flatten())[1]} Good")

print(f"\n2. MODEL ARCHITECTURE:")
print(f"   - Input layer: 2 features")
print(f"   - Normalization layer: Standardizes input features")
print(f"   - Hidden layer 1: 64 neurons (ReLU) + Dropout(0.3)")
print(f"   - Hidden layer 2: 32 neurons (ReLU) + Dropout(0.2)")
print(f"   - Hidden layer 3: 16 neurons (ReLU)")
print(f"   - Output layer: 1 neuron (Sigmoid) for binary classification")
print(f"   - Total parameters: {model.count_params()}")

print(f"\n3. TRAINING CONFIGURATION:")
print(f"   - Optimizer: Adam (learning rate: 0.01)")
print(f"   - Loss function: Binary Crossentropy")
print(f"   - Batch size: 16")
print(f"   - Epochs trained: {len(history.history['loss'])}")
print(f"   - Callbacks: EarlyStopping, ReduceLROnPlateau")

print(f"\n4. TEST SET PERFORMANCE:")
print(f"   - Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   - Precision: {test_precision:.4f}")
print(f"   - Recall: {test_recall:.4f}")
print(f"   - F1-Score: {f1:.4f}")
print(f"   - ROC-AUC: {roc_auc:.4f}")

print(f"\n5. KEY IMPROVEMENTS MADE:")
print(f"   ✓ Fixed undefined variable issue (y extraction)")
print(f"   ✓ Removed artificial 1000x data duplication")
print(f"   ✓ Added proper train/validation/test split")
print(f"   ✓ Improved model architecture (ReLU, Dropout)")
print(f"   ✓ Added early stopping and learning rate reduction")
print(f"   ✓ Comprehensive evaluation metrics")
print(f"   ✓ Visualization of results")
print(f"   ✓ Prediction examples on new data")

print("\n" + "="*70)

## 10. Save the Model (Optional)

In [ ]:
# Save the trained model
model.save('coffee_roasting_model.h5')
print("Model saved as 'coffee_roasting_model.h5'")

# Save the normalization layer parameters for future use
np.save('normalization_mean.npy', norm_layer.mean.numpy())
np.save('normalization_variance.npy', norm_layer.variance.numpy())
print("Normalization parameters saved.")